In [ ]:
%%sql -r dataframe_11
USE WAREHOUSE BIGDATA_MZMB_WH;
USE DATABASE BIGDATA_TAXI_MZMB;
USE SCHEMA GOLD;

CREATE OR REPLACE TABLE GOLD.T7_T10_XGB_RESULTS (
    run_id STRING,
    task STRING,
    model_name STRING,
    model_scope STRING,
    dataset STRING,
    feature_set STRING,
    compute_type STRING,
    scaling_label STRING,

    train_split STRING,
    trainer_eval_split STRING,
    metric_split STRING,

    train_sample_pct FLOAT,
    trainer_eval_sample_pct FLOAT,
    metric_sample_pct FLOAT,

    train_rows NUMBER,
    trainer_eval_rows NUMBER,
    metric_rows NUMBER,

    n_estimators NUMBER,
    train_seconds FLOAT,
    predict_seconds FLOAT,

    mae FLOAT,
    rmse FLOAT,
    r2 FLOAT,

    sum_abs_error FLOAT,
    sum_sq_error FLOAT,
    sum_y FLOAT,
    sum_y2 FLOAT,

    peak_process_memory_mb FLOAT,
    peak_gpu_memory_mb FLOAT,

    ray_nodes NUMBER,
    ray_resources STRING,

    warehouse_name STRING,
    warehouse_size STRING,

    notes STRING,
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE OR REPLACE TABLE GOLD.T7_T10_XGB_FEATURE_IMPORTANCE (
    run_id STRING,
    model_name STRING,
    feature_set STRING,
    feature_name STRING,
    importance_type STRING,
    importance_value FLOAT,
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

In [ ]:
import os
import time
import uuid
import math
import threading
import subprocess

import numpy as np
import pandas as pd
import xgboost as xgb

from snowflake.snowpark.context import get_active_session
from snowflake.ml.data.data_connector import DataConnector
from snowflake.ml.modeling.distributors.xgboost import XGBEstimator, XGBScalingConfig

session = get_active_session()

session.sql("""
    SELECT
        CURRENT_ROLE() AS role,
        CURRENT_WAREHOUSE() AS warehouse,
        CURRENT_DATABASE() AS database_name,
        CURRENT_SCHEMA() AS schema_name
""").show()

In [ ]:
import ray
from snowflake.ml.runtime_cluster import scale_cluster, get_nodes, get_ray_dashboard_url

ray.init(address="auto", ignore_reinit_error=True)

def print_ray_status():
    print("Current Ray nodes:", len(get_nodes()))
    print("Ray cluster resources:", ray.cluster_resources())
    print("Ray available resources:", ray.available_resources())

    try:
        print("Ray dashboard:", get_ray_dashboard_url())
    except Exception as e:
        print("Dashboard unavailable:", e)


def scale_ray_cluster_to(n_nodes):
    print(f"Scaling Ray cluster to {n_nodes} node(s)...")

    scale_cluster(
        expected_cluster_size=n_nodes,
        options={
            "rollback_after_seconds": 1800,
            "block_until_min_cluster_size": n_nodes,
        }
    )

    print_ray_status()


print_ray_status()

In [ ]:
TABLE = "GOLD.T7_DEMAND_FEATURES"

TARGET_LOG = "LOG_TRIP_COUNT"
TARGET_RAW = "TRIP_COUNT"

BASELINE_FEATURES = [
    "DATASET_ID",
    "PICKUP_LOCATION_ID",
    "HOUR_SIN",
    "HOUR_COS",
    "DOW_SIN",
    "DOW_COS",
    "MONTH_SIN",
    "MONTH_COS",
    "IS_WEEKEND",
    "IS_RUSH_HOUR",
    "IS_NIGHT",
    "LAG_1H_TRIP_COUNT_FILLED",
    "LAG_24H_TRIP_COUNT_FILLED",
    "LAG_168H_TRIP_COUNT_FILLED",
    "ROLLING_24H_AVG_TRIP_COUNT_FILLED",
    "ROLLING_168H_AVG_TRIP_COUNT_FILLED",
]

AUGMENTED_FEATURES = BASELINE_FEATURES + [
    "BOROUGH_ID",
    "SERVICE_ZONE_ID",
    "TEMP_MAX_C",
    "TEMP_MIN_C",
    "PRECIPITATION_MM",
    "WEATHER_MISSING",
    "SCHOOL_COUNT",
    "ATTRACTION_COUNT",
    "ACTIVE_EVENTS",
]

In [ ]:
def get_process_memory_mb():
    try:
        import psutil
        process = psutil.Process(os.getpid())
        return process.memory_info().rss / 1024**2
    except Exception:
        return None


def get_gpu_memory_mb():
    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=memory.used",
                "--format=csv,noheader,nounits"
            ],
            capture_output=True,
            text=True,
            timeout=5
        )

        values = []
        for line in result.stdout.strip().splitlines():
            line = line.strip()
            if line:
                values.append(float(line))

        return max(values) if values else None

    except Exception:
        return None


class ResourceMonitor:
    def __init__(self, interval=1.0):
        self.interval = interval
        self.running = False
        self.peak_process_memory_mb = 0.0
        self.peak_gpu_memory_mb = 0.0
        self.thread = None

    def _monitor(self):
        while self.running:
            mem = get_process_memory_mb()
            gpu = get_gpu_memory_mb()

            if mem is not None:
                self.peak_process_memory_mb = max(self.peak_process_memory_mb, mem)

            if gpu is not None:
                self.peak_gpu_memory_mb = max(self.peak_gpu_memory_mb, gpu)

            time.sleep(self.interval)

    def __enter__(self):
        self.running = True
        self.thread = threading.Thread(target=self._monitor, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.running = False
        if self.thread:
            self.thread.join(timeout=2)

In [ ]:
RESULT_COLUMNS_NO_CREATED = [
    "RUN_ID",
    "TASK",
    "MODEL_NAME",
    "MODEL_SCOPE",
    "DATASET",
    "FEATURE_SET",
    "COMPUTE_TYPE",
    "SCALING_LABEL",

    "TRAIN_SPLIT",
    "TRAINER_EVAL_SPLIT",
    "METRIC_SPLIT",

    "TRAIN_SAMPLE_PCT",
    "TRAINER_EVAL_SAMPLE_PCT",
    "METRIC_SAMPLE_PCT",

    "TRAIN_ROWS",
    "TRAINER_EVAL_ROWS",
    "METRIC_ROWS",

    "N_ESTIMATORS",
    "TRAIN_SECONDS",
    "PREDICT_SECONDS",

    "MAE",
    "RMSE",
    "R2",

    "SUM_ABS_ERROR",
    "SUM_SQ_ERROR",
    "SUM_Y",
    "SUM_Y2",

    "PEAK_PROCESS_MEMORY_MB",
    "PEAK_GPU_MEMORY_MB",

    "RAY_NODES",
    "RAY_RESOURCES",

    "WAREHOUSE_NAME",
    "WAREHOUSE_SIZE",

    "NOTES",
]


def sql_value(value):
    if value is None:
        return "NULL"

    if isinstance(value, (float, np.floating)):
        if math.isnan(float(value)) or math.isinf(float(value)):
            return "NULL"
        return str(float(value))

    if isinstance(value, (int, np.integer)):
        return str(int(value))

    text = str(value)
    text = text.replace("'", "''")
    return f"'{text}'"


def save_xgb_result(result):
    cols = ", ".join(RESULT_COLUMNS_NO_CREATED)
    vals = ", ".join(sql_value(result.get(c)) for c in RESULT_COLUMNS_NO_CREATED)

    session.sql(f"""
        INSERT INTO GOLD.T7_T10_XGB_RESULTS ({cols})
        VALUES ({vals})
    """).collect()


def save_feature_importance(run_id, model_name, feature_set, booster):
    importance = booster.get_score(importance_type="gain")

    rows = []
    for feature_name, value in importance.items():
        rows.append({
            "RUN_ID": run_id,
            "MODEL_NAME": model_name,
            "FEATURE_SET": feature_set,
            "FEATURE_NAME": feature_name,
            "IMPORTANCE_TYPE": "gain",
            "IMPORTANCE_VALUE": float(value),
        })

    if rows:
        session.create_dataframe(rows).write.mode("append").save_as_table(
            "GOLD.T7_T10_XGB_FEATURE_IMPORTANCE"
        )

In [ ]:
def get_warehouse_info():
    warehouse_name = session.sql("SELECT CURRENT_WAREHOUSE()").collect()[0][0]

    warehouse_size = None
    try:
        rows = session.sql(f"SHOW WAREHOUSES LIKE '{warehouse_name}'").collect()
        if rows:
            d = rows[0].as_dict()
            warehouse_size = d.get("size") or d.get("SIZE")
    except Exception:
        pass

    return warehouse_name, warehouse_size


def get_ray_metadata():
    try:
        ray_nodes = len(get_nodes())
    except Exception:
        ray_nodes = None

    try:
        ray_resources = str(ray.cluster_resources())
    except Exception:
        ray_resources = None

    return ray_nodes, ray_resources

In [ ]:
def sample_condition_sql(sample_pct):
    if sample_pct is None or sample_pct >= 100:
        return ""

    threshold = int(sample_pct * 100)  # 1% -> 100 out of 10000

    return f"""
        AND MOD(ABS(HASH(DATASET, PICKUP_LOCATION_ID, PICKUP_HOUR)), 10000) < {threshold}
    """


def dataset_filter_sql(dataset):
    if dataset is None:
        return ""
    return f"AND DATASET = '{dataset}'"


def make_train_sdf(features, dataset=None, train_sample_pct=100.0):
    cols = features + [TARGET_LOG]

    sql = f"""
        SELECT {", ".join(cols)}
        FROM {TABLE}
        WHERE SPLIT = 'train'
        {dataset_filter_sql(dataset)}
        {sample_condition_sql(train_sample_pct)}
    """

    return session.sql(sql)


def make_trainer_eval_sdf(features, dataset=None, trainer_eval_sample_pct=1.0):
    cols = features + [TARGET_LOG]

    sql = f"""
        SELECT {", ".join(cols)}
        FROM {TABLE}
        WHERE SPLIT = 'validation'
        {dataset_filter_sql(dataset)}
        {sample_condition_sql(trainer_eval_sample_pct)}
    """

    return session.sql(sql)


def make_metric_sdf(features, split="test", dataset=None, metric_sample_pct=100.0):
    cols = features + [TARGET_LOG, TARGET_RAW]

    sql = f"""
        SELECT {", ".join(cols)}
        FROM {TABLE}
        WHERE SPLIT = '{split}'
        {dataset_filter_sql(dataset)}
        {sample_condition_sql(metric_sample_pct)}
    """

    return session.sql(sql)

In [ ]:
def compute_metrics_from_sums(n, sum_abs_error, sum_sq_error, sum_y, sum_y2):
    mae = sum_abs_error / n
    rmse = (sum_sq_error / n) ** 0.5

    y_mean = sum_y / n
    ss_tot = sum_y2 - n * (y_mean ** 2)

    r2 = 1.0 - (sum_sq_error / ss_tot) if ss_tot > 0 else None

    return mae, rmse, r2


def evaluate_xgb_batched(
    booster,
    features,
    split="test",
    dataset=None,
    metric_sample_pct=100.0
):
    sdf = make_metric_sdf(
        features=features,
        split=split,
        dataset=dataset,
        metric_sample_pct=metric_sample_pct
    )

    n = 0
    sum_abs_error = 0.0
    sum_sq_error = 0.0
    sum_y = 0.0
    sum_y2 = 0.0

    start = time.time()

    for pdf in sdf.to_pandas_batches():
        X_df = pdf[features].astype("float32")
        y_true = pdf[TARGET_RAW].astype("float64").values

        dmatrix = xgb.DMatrix(X_df, feature_names=features)
        pred_log = booster.predict(dmatrix)

        pred_log = np.clip(pred_log, 0, np.log1p(100000))
        pred_raw = np.expm1(pred_log)
        pred_raw = np.clip(pred_raw, 0, None)

        errors = y_true - pred_raw

        n_batch = len(y_true)
        n += n_batch
        sum_abs_error += float(np.abs(errors).sum())
        sum_sq_error += float((errors ** 2).sum())
        sum_y += float(y_true.sum())
        sum_y2 += float((y_true ** 2).sum())

    predict_seconds = time.time() - start

    if n == 0:
        raise ValueError("No rows were evaluated. Check split/sample/dataset filters.")

    mae, rmse, r2 = compute_metrics_from_sums(
        n,
        sum_abs_error,
        sum_sq_error,
        sum_y,
        sum_y2
    )

    return {
        "metric_rows": int(n),
        "predict_seconds": float(predict_seconds),
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": None if r2 is None else float(r2),
        "sum_abs_error": float(sum_abs_error),
        "sum_sq_error": float(sum_sq_error),
        "sum_y": float(sum_y),
        "sum_y2": float(sum_y2),
    }

In [ ]:
def run_snowflake_xgb(
    features,
    feature_set_name,
    use_gpu,
    dataset=None,
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="final",
    n_estimators=30,
    notes=""
):
    run_id = str(uuid.uuid4())

    model_scope = "global" if dataset is None else "separate"
    dataset_label = "all" if dataset is None else dataset

    task = "T10" if use_gpu else "T7"
    model_name = "Snowflake_Distributed_XGBoost_GPU" if use_gpu else "Snowflake_Distributed_XGBoost_CPU"
    compute_type = "Snowflake Ray GPU trainer" if use_gpu else "Snowflake Ray CPU trainer"

    train_sdf = make_train_sdf(
        features=features,
        dataset=dataset,
        train_sample_pct=train_sample_pct
    )

    trainer_eval_sdf = make_trainer_eval_sdf(
        features=features,
        dataset=dataset,
        trainer_eval_sample_pct=trainer_eval_sample_pct
    )

    train_rows = train_sdf.count()
    trainer_eval_rows = trainer_eval_sdf.count()

    print(f"Train rows: {train_rows:,}")
    print(f"Validation rows used by trainer: {trainer_eval_rows:,}")

    train_connector = DataConnector.from_dataframe(train_sdf)
    eval_connector = DataConnector.from_dataframe(trainer_eval_sdf)

    params = {
        "n_estimators": n_estimators,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "max_depth": 6,
        "learning_rate": 0.08,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    }

    estimator = XGBEstimator(
        params=params,
        scaling_config=XGBScalingConfig(use_gpu=use_gpu)
    )

    warehouse_name, warehouse_size = get_warehouse_info()
    ray_nodes, ray_resources = get_ray_metadata()

    with ResourceMonitor(interval=1.0) as monitor:
        start_train = time.time()

        booster = estimator.fit(
            dataset=train_connector,
            input_cols=features,
            label_col=TARGET_LOG,
            eval_set=eval_connector,
            verbose_eval=10
        )

        train_seconds = time.time() - start_train

    if booster is None:
        booster = estimator.get_booster()

    metric_result = evaluate_xgb_batched(
        booster=booster,
        features=features,
        split=metric_split,
        dataset=dataset,
        metric_sample_pct=metric_sample_pct
    )

    result = {
        "RUN_ID": run_id,
        "TASK": task,
        "MODEL_NAME": model_name,
        "MODEL_SCOPE": model_scope,
        "DATASET": dataset_label,
        "FEATURE_SET": feature_set_name,
        "COMPUTE_TYPE": compute_type,
        "SCALING_LABEL": scaling_label,

        "TRAIN_SPLIT": "train",
        "TRAINER_EVAL_SPLIT": "validation",
        "METRIC_SPLIT": metric_split,

        "TRAIN_SAMPLE_PCT": float(train_sample_pct),
        "TRAINER_EVAL_SAMPLE_PCT": float(trainer_eval_sample_pct),
        "METRIC_SAMPLE_PCT": float(metric_sample_pct),

        "TRAIN_ROWS": int(train_rows),
        "TRAINER_EVAL_ROWS": int(trainer_eval_rows),
        "METRIC_ROWS": int(metric_result["metric_rows"]),

        "N_ESTIMATORS": int(n_estimators),
        "TRAIN_SECONDS": float(train_seconds),
        "PREDICT_SECONDS": float(metric_result["predict_seconds"]),

        "MAE": float(metric_result["mae"]),
        "RMSE": float(metric_result["rmse"]),
        "R2": None if metric_result["r2"] is None else float(metric_result["r2"]),

        "SUM_ABS_ERROR": float(metric_result["sum_abs_error"]),
        "SUM_SQ_ERROR": float(metric_result["sum_sq_error"]),
        "SUM_Y": float(metric_result["sum_y"]),
        "SUM_Y2": float(metric_result["sum_y2"]),

        "PEAK_PROCESS_MEMORY_MB": float(monitor.peak_process_memory_mb),
        "PEAK_GPU_MEMORY_MB": float(monitor.peak_gpu_memory_mb),

        "RAY_NODES": int(ray_nodes) if ray_nodes is not None else None,
        "RAY_RESOURCES": ray_resources,

        "WAREHOUSE_NAME": warehouse_name,
        "WAREHOUSE_SIZE": warehouse_size,

        "NOTES": notes,
    }

    save_xgb_result(result)
    # save_feature_importance(run_id, model_name, feature_set_name, booster)

    print("\nSaved result:")
    print(result)

    print("\nFeature importance:")
    print(booster.get_score(importance_type="gain"))

    return booster, result

In [ ]:
def save_combined_separate_result(results, scaling_label, feature_set_name, use_gpu=True, notes=""):
    total_rows = sum(r["METRIC_ROWS"] for r in results)
    total_abs = sum(r["SUM_ABS_ERROR"] for r in results)
    total_sq = sum(r["SUM_SQ_ERROR"] for r in results)
    total_y = sum(r["SUM_Y"] for r in results)
    total_y2 = sum(r["SUM_Y2"] for r in results)

    mae, rmse, r2 = compute_metrics_from_sums(
        total_rows,
        total_abs,
        total_sq,
        total_y,
        total_y2
    )

    train_rows = sum(r["TRAIN_ROWS"] for r in results)
    trainer_eval_rows = sum(r["TRAINER_EVAL_ROWS"] for r in results)
    train_seconds = sum(r["TRAIN_SECONDS"] for r in results)
    predict_seconds = sum(r["PREDICT_SECONDS"] for r in results)

    warehouse_name, warehouse_size = get_warehouse_info()
    ray_nodes, ray_resources = get_ray_metadata()

    result = {
        "RUN_ID": str(uuid.uuid4()),
        "TASK": "T10" if use_gpu else "T7",
        "MODEL_NAME": "Combined_Separate_XGBoost_Models",
        "MODEL_SCOPE": "separate_combined",
        "DATASET": "yellow+green+fhv+fhvhv",
        "FEATURE_SET": feature_set_name,
        "COMPUTE_TYPE": "Combined metrics from four separate XGBoost models",
        "SCALING_LABEL": scaling_label,

        "TRAIN_SPLIT": "train",
        "TRAINER_EVAL_SPLIT": "validation",
        "METRIC_SPLIT": "test",

        "TRAIN_SAMPLE_PCT": results[0]["TRAIN_SAMPLE_PCT"],
        "TRAINER_EVAL_SAMPLE_PCT": results[0]["TRAINER_EVAL_SAMPLE_PCT"],
        "METRIC_SAMPLE_PCT": results[0]["METRIC_SAMPLE_PCT"],

        "TRAIN_ROWS": int(train_rows),
        "TRAINER_EVAL_ROWS": int(trainer_eval_rows),
        "METRIC_ROWS": int(total_rows),

        "N_ESTIMATORS": results[0]["N_ESTIMATORS"],
        "TRAIN_SECONDS": float(train_seconds),
        "PREDICT_SECONDS": float(predict_seconds),

        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": None if r2 is None else float(r2),

        "SUM_ABS_ERROR": float(total_abs),
        "SUM_SQ_ERROR": float(total_sq),
        "SUM_Y": float(total_y),
        "SUM_Y2": float(total_y2),

        "PEAK_PROCESS_MEMORY_MB": max(r["PEAK_PROCESS_MEMORY_MB"] for r in results),
        "PEAK_GPU_MEMORY_MB": max(r["PEAK_GPU_MEMORY_MB"] for r in results),

        "RAY_NODES": int(ray_nodes) if ray_nodes is not None else None,
        "RAY_RESOURCES": ray_resources,

        "WAREHOUSE_NAME": warehouse_name,
        "WAREHOUSE_SIZE": warehouse_size,

        "NOTES": notes,
    }

    save_xgb_result(result)

    print("\nSaved combined separate-model result:")
    print(result)

    return result

In [ ]:
scale_ray_cluster_to(1)

In [ ]:
booster_base_1node, result_base_1node = run_snowflake_xgb(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    use_gpu=True,
    dataset=None,
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_baseline_1_node_full_train_full_test",
    n_estimators=30,
    notes="Baseline XGBoost GPU run without T5 augmentation. Full train and full test. 1 Ray node."
)

In [ ]:
scale_ray_cluster_to(2)

In [ ]:
booster_base_2nodes, result_base_2nodes = run_snowflake_xgb(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    use_gpu=True,
    dataset=None,
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_baseline_2_nodes_full_train_full_test",
    n_estimators=30,
    notes="Baseline XGBoost GPU run without T5 augmentation. Full train and full test. 2 Ray nodes."
)

In [ ]:
scale_ray_cluster_to(4)

In [ ]:
booster_base_4nodes, result_base_4nodes = run_snowflake_xgb(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    use_gpu=True,
    dataset=None,
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_baseline_4_nodes_full_train_full_test",
    n_estimators=30,
    notes="Baseline XGBoost GPU run without T5 augmentation. Full train and full test. 4 Ray nodes."
)

In [ ]:
scale_ray_cluster_to(2)

In [ ]:
booster_aug_2nodes, result_aug_2nodes = run_snowflake_xgb(
    features=AUGMENTED_FEATURES,
    feature_set_name="augmented_with_t5",
    use_gpu=True,
    dataset=None,
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_augmented_2_nodes_full_train_full_test",
    n_estimators=30,
    notes="Augmented XGBoost GPU run with T5 features. Full train and full test. 2 Ray nodes."
)

In [ ]:
scale_ray_cluster_to(4)

In [ ]:
booster_aug_2nodes, result_aug_2nodes = run_snowflake_xgb(
    features=AUGMENTED_FEATURES,
    feature_set_name="augmented_with_t5",
    use_gpu=True,
    dataset=None,
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_augmented_4_nodes_full_train_full_test",
    n_estimators=30,
    notes="Augmented XGBoost GPU run with T5 features. Full train and full test. 4 Ray nodes."
)

In [ ]:
scale_ray_cluster_to(4)

In [ ]:
separate_results = []

In [ ]:
booster_yellow, result_yellow = run_snowflake_xgb(
    features=AUGMENTED_FEATURES,
    feature_set_name="augmented_with_t5",
    use_gpu=True,
    dataset="yellow",
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_separate_yellow_4_nodes_full_train_full_test",
    n_estimators=30,
    notes="Separate Yellow XGBoost GPU model with T5 features. Full train and full test. 4 Ray nodes."
)
separate_results.append(result_yellow)

In [ ]:
booster_green, result_green = run_snowflake_xgb(
    features=AUGMENTED_FEATURES,
    feature_set_name="augmented_with_t5",
    use_gpu=True,
    dataset="green",
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_separate_green_4_nodes_full_train_full_test",
    n_estimators=30,
    notes="Separate Green XGBoost GPU model with T5 features. Full train and full test. 4 Ray nodes."
)
separate_results.append(result_green)

In [ ]:
booster_fhv, result_fhv = run_snowflake_xgb(
    features=AUGMENTED_FEATURES,
    feature_set_name="augmented_with_t5",
    use_gpu=True,
    dataset="fhv",
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_separate_fhv_4_nodes_full_train_full_test",
    n_estimators=30,
    notes="Separate FHV XGBoost GPU model with T5 features. Full train and full test. 4 Ray nodes."
)
separate_results.append(result_fhv)

In [ ]:
booster_fhvhv, result_fhvhv = run_snowflake_xgb(
    features=AUGMENTED_FEATURES,
    feature_set_name="augmented_with_t5",
    use_gpu=True,
    dataset="fhvhv",
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_gpu_separate_fhvhv_4_nodes_full_train_full_test",
    n_estimators=30,
    notes="Separate FHVHV XGBoost GPU model with T5 features. Full train and full test. 4 Ray nodes."
)
separate_results.append(result_fhvhv)

In [ ]:
combined_separate_result = save_combined_separate_result(
    separate_results,
    scaling_label="xgb_gpu_combined_separate_4_nodes_full_train_full_test",
    feature_set_name="augmented_with_t5",
    use_gpu=True,
    notes="Combined metrics from four separate dataset-specific XGBoost GPU models."
)

In [ ]:
%%sql -r dataframe_2
SELECT
    task,
    model_name,
    model_scope,
    dataset,
    feature_set,
    scaling_label,
    train_rows,
    metric_rows,
    ray_nodes,
    n_estimators,
    train_seconds,
    predict_seconds,
    mae,
    rmse,
    r2,
    peak_process_memory_mb,
    peak_gpu_memory_mb,
    created_at
FROM GOLD.T7_T10_XGB_RESULTS
ORDER BY created_at;

In [ ]:
scale_ray_cluster_to(4)

In [ ]:
booster_aug_cpu_4nodes, result_aug_cpu_4nodes = run_snowflake_xgb(
    features=AUGMENTED_FEATURES,
    feature_set_name="augmented_with_t5",
    use_gpu=False,
    dataset=None,
    train_sample_pct=100.0,
    trainer_eval_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="xgb_cpu_augmented_4_nodes_full_train_full_test",
    n_estimators=30,
    notes="CPU distributed XGBoost run with T5 augmentation. Full train and full test. 4 Ray CPU nodes."
)

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

session.sql("USE WAREHOUSE BIGDATA_MZMB_WH").collect()
session.sql("USE DATABASE BIGDATA_TAXI_MZMB").collect()
session.sql("USE SCHEMA GOLD").collect()

session.sql("""
    SELECT
        CURRENT_ROLE() AS role,
        CURRENT_WAREHOUSE() AS warehouse,
        CURRENT_DATABASE() AS database_name,
        CURRENT_SCHEMA() AS schema_name
""").show()


In [ ]:
session.sql("""
CREATE STAGE IF NOT EXISTS GOLD.T7_MODEL_STAGE
""").collect()

session.sql("""
CREATE TABLE IF NOT EXISTS GOLD.T7_T10_PYTORCH_RESULTS (
    run_id STRING,
    task STRING,
    model_name STRING,
    model_scope STRING,
    dataset STRING,
    feature_set STRING,
    scaling_label STRING,

    compute_type STRING,
    use_gpu BOOLEAN,
    num_nodes NUMBER,
    workers_per_node NUMBER,
    total_workers NUMBER,
    epochs NUMBER,
    batch_size NUMBER,

    train_split STRING,
    metric_split STRING,
    train_sample_pct FLOAT,
    metric_sample_pct FLOAT,

    train_rows NUMBER,
    metric_rows NUMBER,

    train_seconds FLOAT,
    predict_seconds FLOAT,

    train_loss FLOAT,
    mae FLOAT,
    rmse FLOAT,
    r2 FLOAT,
    log_rmse FLOAT,

    sum_abs_error FLOAT,
    sum_sq_error FLOAT,
    sum_y FLOAT,
    sum_y2 FLOAT,

    peak_process_memory_mb FLOAT,
    peak_gpu_memory_mb FLOAT,

    artifact_location STRING,
    notes STRING,
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
""").collect()

session.sql("""
CREATE TABLE IF NOT EXISTS GOLD.T7_SGD_RESULTS (
    run_id STRING,
    task STRING,
    model_name STRING,
    model_scope STRING,
    dataset STRING,
    feature_set STRING,
    scaling_label STRING,

    compute_type STRING,
    train_split STRING,
    metric_split STRING,
    train_sample_pct FLOAT,
    metric_sample_pct FLOAT,

    train_rows NUMBER,
    metric_rows NUMBER,

    train_seconds FLOAT,
    predict_seconds FLOAT,

    train_loss FLOAT,
    mae FLOAT,
    rmse FLOAT,
    r2 FLOAT,
    log_rmse FLOAT,

    sum_abs_error FLOAT,
    sum_sq_error FLOAT,
    sum_y FLOAT,
    sum_y2 FLOAT,

    peak_process_memory_mb FLOAT,
    peak_gpu_memory_mb FLOAT,

    warehouse_name STRING,
    warehouse_size STRING,

    notes STRING,
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
""").collect()

In [ ]:
import os
import json
import time
import uuid
import math
import tempfile
import glob
import threading
import subprocess

import numpy as np
import pandas as pd

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU count:", torch.cuda.device_count())
        print("GPU name:", torch.cuda.get_device_name(0))
except Exception as e:
    print("PyTorch import failed:")
    print(repr(e))

try:
    from snowflake.ml.data.sharded_data_connector import ShardedDataConnector
    from snowflake.ml.modeling.distributors.pytorch import (
        PyTorchDistributor,
        PyTorchScalingConfig,
        WorkerResourceConfig,
        get_context,
    )
    print("Snowflake PyTorch Distributor available.")
except Exception as e:
    print("Snowflake PyTorch Distributor import failed:")
    print(repr(e))

try:
    from sklearn.linear_model import SGDRegressor
    print("sklearn SGDRegressor available.")
except Exception as e:
    print("sklearn import failed:")
    print(repr(e))


In [ ]:
TABLE = "GOLD.T7_DEMAND_FEATURES"

TARGET_LOG = "LOG_TRIP_COUNT"
TARGET_RAW = "TRIP_COUNT"

BASELINE_FEATURES = [
    "DATASET_ID",
    "PICKUP_LOCATION_ID",
    "HOUR_SIN",
    "HOUR_COS",
    "DOW_SIN",
    "DOW_COS",
    "MONTH_SIN",
    "MONTH_COS",
    "IS_WEEKEND",
    "IS_RUSH_HOUR",
    "IS_NIGHT",
    "LAG_1H_TRIP_COUNT_FILLED",
    "LAG_24H_TRIP_COUNT_FILLED",
    "LAG_168H_TRIP_COUNT_FILLED",
    "ROLLING_24H_AVG_TRIP_COUNT_FILLED",
    "ROLLING_168H_AVG_TRIP_COUNT_FILLED",
]

AUGMENTED_FEATURES = BASELINE_FEATURES + [
    "BOROUGH_ID",
    "SERVICE_ZONE_ID",
    "TEMP_MAX_C",
    "TEMP_MIN_C",
    "PRECIPITATION_MM",
    "WEATHER_MISSING",
    "SCHOOL_COUNT",
    "ATTRACTION_COUNT",
    "ACTIVE_EVENTS",
]

print("Baseline features:", len(BASELINE_FEATURES))
print("Augmented features:", len(AUGMENTED_FEATURES))


In [ ]:
def sample_condition_sql(sample_pct):
    if sample_pct is None or sample_pct >= 100:
        return ""

    threshold = int(sample_pct * 100)  # percent into 0..10000 hash bucket

    return f"""
        AND MOD(ABS(HASH(DATASET, PICKUP_LOCATION_ID, PICKUP_HOUR)), 10000) < {threshold}
    """


def dataset_filter_sql(dataset):
    if dataset is None:
        return ""
    return f"AND DATASET = '{dataset}'"


def get_scaler_stats_sql(features, dataset=None, train_sample_pct=100.0):
    mean_exprs = [
        f"AVG(COALESCE({f}, 0)) AS MEAN_{i}"
        for i, f in enumerate(features)
    ]

    std_exprs = [
        f"STDDEV_POP(COALESCE({f}, 0)) AS STD_{i}"
        for i, f in enumerate(features)
    ]

    sql = f"""
        SELECT
            {", ".join(mean_exprs + std_exprs)}
        FROM {TABLE}
        WHERE SPLIT = 'train'
        {dataset_filter_sql(dataset)}
        {sample_condition_sql(train_sample_pct)}
    """

    row = session.sql(sql).collect()[0]
    values = list(row)

    n_features = len(features)

    means = np.array(
        [float(v) if v is not None else 0.0 for v in values[:n_features]],
        dtype=np.float32,
    )

    stds = np.array(
        [float(v) if v is not None else 1.0 for v in values[n_features:]],
        dtype=np.float32,
    )

    stds = np.where((stds == 0) | np.isnan(stds), 1.0, stds).astype(np.float32)

    return means, stds


def compute_metrics_from_sums(n, sum_abs_error, sum_sq_error, sum_y, sum_y2):
    mae = sum_abs_error / n
    rmse = (sum_sq_error / n) ** 0.5

    y_mean = sum_y / n
    ss_tot = sum_y2 - n * (y_mean ** 2)

    r2 = 1.0 - (sum_sq_error / ss_tot) if ss_tot > 0 else None

    return mae, rmse, r2


def sql_value(value):
    if value is None:
        return "NULL"

    if isinstance(value, bool):
        return "TRUE" if value else "FALSE"

    if isinstance(value, (float, np.floating)):
        if math.isnan(float(value)) or math.isinf(float(value)):
            return "NULL"
        return str(float(value))

    if isinstance(value, (int, np.integer)):
        return str(int(value))

    text = str(value)
    text = text.replace("'", "''")
    return f"'{text}'"


def get_process_memory_mb():
    try:
        import psutil
        process = psutil.Process(os.getpid())
        return process.memory_info().rss / 1024**2
    except Exception:
        return None


def get_gpu_memory_mb():
    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=memory.used",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            timeout=5,
        )

        values = []
        for line in result.stdout.strip().splitlines():
            line = line.strip()
            if line:
                values.append(float(line))

        return max(values) if values else 0.0

    except Exception:
        return 0.0


class ResourceMonitor:
    def __init__(self, interval=1.0):
        self.interval = interval
        self.running = False
        self.peak_process_memory_mb = 0.0
        self.peak_gpu_memory_mb = 0.0
        self.thread = None

    def _monitor(self):
        while self.running:
            mem = get_process_memory_mb()
            gpu = get_gpu_memory_mb()

            if mem is not None:
                self.peak_process_memory_mb = max(self.peak_process_memory_mb, mem)

            if gpu is not None:
                self.peak_gpu_memory_mb = max(self.peak_gpu_memory_mb, gpu)

            time.sleep(self.interval)

    def __enter__(self):
        self.running = True
        self.thread = threading.Thread(target=self._monitor, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.running = False
        if self.thread:
            self.thread.join(timeout=2)


def get_warehouse_info():
    warehouse_name = session.sql("SELECT CURRENT_WAREHOUSE()").collect()[0][0]

    warehouse_size = None
    try:
        rows = session.sql(f"SHOW WAREHOUSES LIKE '{warehouse_name}'").collect()
        if rows:
            d = rows[0].as_dict()
            warehouse_size = d.get("size") or d.get("SIZE")
    except Exception:
        pass

    return warehouse_name, warehouse_size


In [ ]:
def make_pytorch_sdf(features, split, dataset=None, sample_pct=100.0):
    # Cast all columns to FLOAT so PyTorch receives clean numeric tensors.
    select_exprs = []

    for f in features:
        select_exprs.append(f"COALESCE({f}, 0)::FLOAT AS {f}")

    select_exprs.append(f"COALESCE({TARGET_LOG}, 0)::FLOAT AS {TARGET_LOG}")

    if split != "train":
        select_exprs.append(f"COALESCE({TARGET_RAW}, 0)::FLOAT AS {TARGET_RAW}")

    sql = f"""
        SELECT
            {", ".join(select_exprs)}
        FROM {TABLE}
        WHERE SPLIT = '{split}'
        {dataset_filter_sql(dataset)}
        {sample_condition_sql(sample_pct)}
    """

    return session.sql(sql)


PYTORCH_RESULT_COLUMNS_NO_CREATED = [
    "RUN_ID",
    "TASK",
    "MODEL_NAME",
    "MODEL_SCOPE",
    "DATASET",
    "FEATURE_SET",
    "SCALING_LABEL",

    "COMPUTE_TYPE",
    "USE_GPU",
    "NUM_NODES",
    "WORKERS_PER_NODE",
    "TOTAL_WORKERS",
    "EPOCHS",
    "BATCH_SIZE",

    "TRAIN_SPLIT",
    "METRIC_SPLIT",
    "TRAIN_SAMPLE_PCT",
    "METRIC_SAMPLE_PCT",

    "TRAIN_ROWS",
    "METRIC_ROWS",

    "TRAIN_SECONDS",
    "PREDICT_SECONDS",

    "TRAIN_LOSS",
    "MAE",
    "RMSE",
    "R2",
    "LOG_RMSE",

    "SUM_ABS_ERROR",
    "SUM_SQ_ERROR",
    "SUM_Y",
    "SUM_Y2",

    "PEAK_PROCESS_MEMORY_MB",
    "PEAK_GPU_MEMORY_MB",

    "ARTIFACT_LOCATION",
    "NOTES",
]


def save_pytorch_result(result):
    cols = ", ".join(PYTORCH_RESULT_COLUMNS_NO_CREATED)
    vals = ", ".join(sql_value(result.get(c)) for c in PYTORCH_RESULT_COLUMNS_NO_CREATED)

    session.sql(f"""
        INSERT INTO GOLD.T7_T10_PYTORCH_RESULTS ({cols})
        VALUES ({vals})
    """).collect()


def load_json_from_model_dir(model_dir, filename="metrics.json"):
    if model_dir is None:
        return None

    # Case 1: local path
    local_path = os.path.join(model_dir, filename)
    if os.path.exists(local_path):
        with open(local_path, "r") as f:
            return json.load(f)

    # Case 2: Snowflake stage path
    stage_path = model_dir

    if not stage_path.startswith("@"):
        stage_path = "@" + stage_path

    stage_path = stage_path.rstrip("/")

    local_dir = tempfile.mkdtemp()

    try:
        session.file.get(f"{stage_path}/{filename}", local_dir)

        matches = glob.glob(os.path.join(local_dir, "**", filename), recursive=True)

        if not matches:
            print("Could not find downloaded metrics.json.")
            print("Downloaded files:", glob.glob(os.path.join(local_dir, "**", "*"), recursive=True))
            return None

        with open(matches[0], "r") as f:
            return json.load(f)

    except Exception as e:
        print("Could not load metrics from model_dir:")
        print("model_dir:", model_dir)
        print("error:", repr(e))
        return None


In [ ]:
def make_pytorch_train_func(
    input_cols,
    label_col,
    raw_label_col,
    mean_values_list,
    std_values_list,
    batch_size,
    epochs,
    hidden_size,
    use_gpu,
):
    def pytorch_train_func():
        import os
        import json
        import time
        import math

        import torch
        import torch.nn as nn
        import torch.optim as optim
        import torch.distributed as dist
        from torch.nn.parallel import DistributedDataParallel as DDP
        from torch.utils.data import DataLoader

        from snowflake.ml.modeling.distributors.pytorch import get_context

        class DemandNet(nn.Module):
            def __init__(self, input_size, hidden_size):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(input_size, hidden_size),
                    nn.ReLU(),
                    nn.Linear(hidden_size, hidden_size),
                    nn.ReLU(),
                    nn.Linear(hidden_size, 1),
                )

            def forward(self, x):
                return self.net(x)

        def process_memory_mb():
            try:
                import psutil
                process = psutil.Process(os.getpid())
                return process.memory_info().rss / 1024**2
            except Exception:
                return 0.0

        def tensor_column(batch_dict, col):
            x = batch_dict[col]

            if not torch.is_tensor(x):
                x = torch.tensor(x)

            x = x.float()

            if x.ndim == 1:
                x = x.unsqueeze(1)
            elif x.ndim == 2:
                if x.shape[0] == 1:
                    x = x.T
                elif x.shape[1] == 1:
                    pass
                else:
                    x = x.reshape(-1, 1)
            else:
                x = x.reshape(-1, 1)

            return x

        def make_features(batch_dict, cols, device):
            x_cols = [tensor_column(batch_dict, col) for col in cols]
            x = torch.cat(x_cols, dim=1)
            return x.to(device)

        def make_label(batch_dict, col, device):
            y = tensor_column(batch_dict, col).squeeze(1)
            return y.to(device)

        context = get_context()
        rank = context.get_rank()
        local_rank = context.get_local_rank()

        if not dist.is_initialized():
            dist.init_process_group(backend="gloo")

        use_cuda = bool(use_gpu) and torch.cuda.is_available()

        if use_cuda:
            device = torch.device(f"cuda:{local_rank}")
            torch.cuda.set_device(device)
        else:
            device = torch.device("cpu")

        mean_values = torch.tensor(mean_values_list, dtype=torch.float32, device=device)
        std_values = torch.tensor(std_values_list, dtype=torch.float32, device=device)

        model = DemandNet(input_size=len(input_cols), hidden_size=hidden_size).to(device)

        if use_cuda:
            ddp_model = DDP(model, device_ids=[local_rank])
        else:
            ddp_model = DDP(model)

        criterion = nn.MSELoss()
        optimizer = optim.Adam(ddp_model.parameters(), lr=0.001)

        dataset_map = context.get_dataset_map()

        train_dataset = dataset_map["train"].get_shard().to_torch_dataset(batch_size=batch_size)
        train_loader = DataLoader(train_dataset)

        train_rows_local = 0
        train_loss_sum_local = 0.0
        train_batches_local = 0

        peak_memory_mb = process_memory_mb()

        if use_cuda:
            torch.cuda.reset_peak_memory_stats(device)

        start_train = time.time()

        ddp_model.train()

        for epoch in range(epochs):
            for batch_dict in train_loader:
                x = make_features(batch_dict, input_cols, device)
                y = make_label(batch_dict, label_col, device)

                x = (x - mean_values) / std_values

                pred = ddp_model(x).squeeze(1)
                loss = criterion(pred, y)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                batch_n = y.shape[0]
                train_rows_local += int(batch_n)
                train_loss_sum_local += float(loss.item()) * int(batch_n)
                train_batches_local += 1

                peak_memory_mb = max(peak_memory_mb, process_memory_mb())

                if rank == 0 and train_batches_local % 100 == 0:
                    print(
                        f"epoch={epoch + 1}/{epochs}, "
                        f"rank={rank}, "
                        f"local_batches={train_batches_local}, "
                        f"local_rows={train_rows_local:,}, "
                        f"loss={loss.item():.5f}"
                    )

        train_seconds_local = time.time() - start_train

        train_tensor = torch.tensor(
            [
                float(train_rows_local),
                float(train_loss_sum_local),
                float(train_seconds_local),
                float(peak_memory_mb),
            ],
            dtype=torch.float64,
        )

        rows_loss_tensor = train_tensor[:2].clone()
        dist.all_reduce(rows_loss_tensor, op=dist.ReduceOp.SUM)

        time_mem_tensor = train_tensor[2:].clone()
        dist.all_reduce(time_mem_tensor, op=dist.ReduceOp.MAX)

        train_rows_total = int(rows_loss_tensor[0].item())
        train_loss_mean = rows_loss_tensor[1].item() / train_rows_total if train_rows_total > 0 else None
        train_seconds_max = time_mem_tensor[0].item()
        peak_memory_max = time_mem_tensor[1].item()

        test_dataset = dataset_map["test"].get_shard().to_torch_dataset(batch_size=batch_size)
        test_loader = DataLoader(test_dataset)

        ddp_model.eval()

        metric_rows_local = 0
        sum_abs_error_local = 0.0
        sum_sq_error_local = 0.0
        sum_y_local = 0.0
        sum_y2_local = 0.0
        sum_log_sq_error_local = 0.0

        start_eval = time.time()

        with torch.no_grad():
            for batch_dict in test_loader:
                x = make_features(batch_dict, input_cols, device)
                y_log = make_label(batch_dict, label_col, device)
                y_raw = make_label(batch_dict, raw_label_col, device)

                x = (x - mean_values) / std_values

                pred_log = ddp_model(x).squeeze(1)

                pred_log = torch.clamp(pred_log, min=0.0, max=math.log1p(100000.0))
                pred_raw = torch.expm1(pred_log)
                pred_raw = torch.clamp(pred_raw, min=0.0)

                errors = y_raw - pred_raw
                log_errors = y_log - pred_log

                metric_rows_local += int(y_raw.shape[0])
                sum_abs_error_local += float(torch.abs(errors).sum().item())
                sum_sq_error_local += float((errors ** 2).sum().item())
                sum_y_local += float(y_raw.sum().item())
                sum_y2_local += float((y_raw ** 2).sum().item())
                sum_log_sq_error_local += float((log_errors ** 2).sum().item())

        predict_seconds_local = time.time() - start_eval

        metric_tensor = torch.tensor(
            [
                float(metric_rows_local),
                sum_abs_error_local,
                sum_sq_error_local,
                sum_y_local,
                sum_y2_local,
                sum_log_sq_error_local,
            ],
            dtype=torch.float64,
        )

        dist.all_reduce(metric_tensor, op=dist.ReduceOp.SUM)

        pred_time_tensor = torch.tensor([float(predict_seconds_local)], dtype=torch.float64)
        dist.all_reduce(pred_time_tensor, op=dist.ReduceOp.MAX)

        metric_rows_total = int(metric_tensor[0].item())
        sum_abs_error = metric_tensor[1].item()
        sum_sq_error = metric_tensor[2].item()
        sum_y = metric_tensor[3].item()
        sum_y2 = metric_tensor[4].item()
        sum_log_sq_error = metric_tensor[5].item()
        predict_seconds_max = pred_time_tensor[0].item()

        mae = sum_abs_error / metric_rows_total
        rmse = math.sqrt(sum_sq_error / metric_rows_total)

        y_mean = sum_y / metric_rows_total
        ss_tot = sum_y2 - metric_rows_total * (y_mean ** 2)
        r2 = 1.0 - (sum_sq_error / ss_tot) if ss_tot > 0 else None

        log_rmse = math.sqrt(sum_log_sq_error / metric_rows_total)

        if use_cuda:
            gpu_mem_mb_local = torch.cuda.max_memory_allocated(device) / 1024**2
        else:
            gpu_mem_mb_local = 0.0

        gpu_tensor = torch.tensor([float(gpu_mem_mb_local)], dtype=torch.float64)
        dist.all_reduce(gpu_tensor, op=dist.ReduceOp.MAX)
        peak_gpu_memory_mb = gpu_tensor[0].item()

        metrics = {
            "train_rows": train_rows_total,
            "metric_rows": metric_rows_total,
            "train_seconds": train_seconds_max,
            "predict_seconds": predict_seconds_max,
            "train_loss": train_loss_mean,
            "mae": mae,
            "rmse": rmse,
            "r2": r2,
            "log_rmse": log_rmse,
            "sum_abs_error": sum_abs_error,
            "sum_sq_error": sum_sq_error,
            "sum_y": sum_y,
            "sum_y2": sum_y2,
            "peak_process_memory_mb": peak_memory_max,
            "peak_gpu_memory_mb": peak_gpu_memory_mb,
        }

        if rank == 0:
            model_dir = context.get_model_dir()
            os.makedirs(model_dir, exist_ok=True)

            torch.save(
                ddp_model.module.state_dict(),
                os.path.join(model_dir, "model.pt"),
            )

            with open(os.path.join(model_dir, "metrics.json"), "w") as f:
                json.dump(metrics, f)

            print("Saved model and metrics to:", model_dir)
            print("Metrics:", metrics)

        dist.barrier()

        if dist.is_initialized():
            dist.destroy_process_group()

        return metrics if rank == 0 else {}

    return pytorch_train_func


In [ ]:
def run_pytorch_ddp(
    features,
    feature_set_name,
    use_gpu,
    num_nodes,
    workers_per_node=1,
    dataset=None,
    train_sample_pct=100.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="pytorch_ddp",
    epochs=1,
    batch_size=4096,
    hidden_size=64,
    notes="",
):
    run_id = str(uuid.uuid4())

    model_scope = "global" if dataset is None else "separate"
    dataset_label = "all" if dataset is None else dataset

    total_workers = int(num_nodes * workers_per_node)

    print("=" * 80)
    print(f"Run: {scaling_label}")
    print(f"Feature set: {feature_set_name}")
    print(f"Use GPU: {use_gpu}")
    print(f"Nodes: {num_nodes}")
    print(f"Workers per node: {workers_per_node}")
    print(f"Total workers: {total_workers}")
    print(f"Epochs: {epochs}")
    print(f"Batch size: {batch_size}")
    print(f"Train sample: {train_sample_pct}%")
    print(f"Metric sample: {metric_sample_pct}%")
    print("=" * 80)

    if use_gpu:
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError(
                "use_gpu=True, but torch.cuda.is_available() is False. "
                "Reconnect the notebook to a GPU runtime first."
            )

    print("Computing scaler statistics...")
    mean_values, std_values = get_scaler_stats_sql(
        features=features,
        dataset=dataset,
        train_sample_pct=train_sample_pct,
    )

    train_sdf = make_pytorch_sdf(
        features=features,
        split="train",
        dataset=dataset,
        sample_pct=train_sample_pct,
    )

    test_sdf = make_pytorch_sdf(
        features=features,
        split=metric_split,
        dataset=dataset,
        sample_pct=metric_sample_pct,
    )

    train_rows_expected = train_sdf.count()
    metric_rows_expected = test_sdf.count()

    print(f"Expected train rows: {train_rows_expected:,}")
    print(f"Expected metric rows: {metric_rows_expected:,}")

    train_connector = ShardedDataConnector.from_dataframe(train_sdf)
    test_connector = ShardedDataConnector.from_dataframe(test_sdf)

    train_func = make_pytorch_train_func(
        input_cols=features,
        label_col=TARGET_LOG,
        raw_label_col=TARGET_RAW,
        mean_values_list=mean_values.astype(float).tolist(),
        std_values_list=std_values.astype(float).tolist(),
        batch_size=int(batch_size),
        epochs=int(epochs),
        hidden_size=int(hidden_size),
        use_gpu=bool(use_gpu),
    )

    if use_gpu:
        compute_type = "Snowflake PyTorch DDP GPU"
        worker_resources = WorkerResourceConfig(num_cpus=1, num_gpus=1)
    else:
        compute_type = "Snowflake PyTorch DDP CPU"
        worker_resources = WorkerResourceConfig(num_cpus=1, num_gpus=0)

    scaling_config = PyTorchScalingConfig(
        num_nodes=int(num_nodes),
        num_workers_per_node=int(workers_per_node),
        resource_requirements_per_worker=worker_resources,
    )

    pytorch_trainer = PyTorchDistributor(
        train_func=train_func,
        scaling_config=scaling_config,
    )

    response = pytorch_trainer.run(
        dataset_map={
            "train": train_connector,
            "test": test_connector,
        },
        artifact_stage_location="BIGDATA_TAXI_MZMB.GOLD.T7_MODEL_STAGE",
    )

    print("PyTorch response:", response)

    model_dir = None
    try:
        model_dir = response.get_model_dir()
    except Exception as e:
        print("Could not get model dir from response:")
        print(repr(e))

    print("Model dir:", model_dir)

    metrics = load_json_from_model_dir(model_dir, filename="metrics.json")

    if metrics is None:
        print("Could not load metrics.json from artifact location.")
        print("Try inspecting response manually:")
        print(dir(response))
        raise RuntimeError("PyTorch run finished, but metrics could not be loaded.")

    result = {
        "RUN_ID": run_id,
        "TASK": "T10" if use_gpu else "T7",
        "MODEL_NAME": "Snowflake_PyTorch_DDP",
        "MODEL_SCOPE": model_scope,
        "DATASET": dataset_label,
        "FEATURE_SET": feature_set_name,
        "SCALING_LABEL": scaling_label,

        "COMPUTE_TYPE": compute_type,
        "USE_GPU": bool(use_gpu),
        "NUM_NODES": int(num_nodes),
        "WORKERS_PER_NODE": int(workers_per_node),
        "TOTAL_WORKERS": int(total_workers),
        "EPOCHS": int(epochs),
        "BATCH_SIZE": int(batch_size),

        "TRAIN_SPLIT": "train",
        "METRIC_SPLIT": metric_split,
        "TRAIN_SAMPLE_PCT": float(train_sample_pct),
        "METRIC_SAMPLE_PCT": float(metric_sample_pct),

        "TRAIN_ROWS": int(metrics["train_rows"]),
        "METRIC_ROWS": int(metrics["metric_rows"]),

        "TRAIN_SECONDS": float(metrics["train_seconds"]),
        "PREDICT_SECONDS": float(metrics["predict_seconds"]),

        "TRAIN_LOSS": None if metrics.get("train_loss") is None else float(metrics["train_loss"]),
        "MAE": float(metrics["mae"]),
        "RMSE": float(metrics["rmse"]),
        "R2": None if metrics.get("r2") is None else float(metrics["r2"]),
        "LOG_RMSE": float(metrics["log_rmse"]),

        "SUM_ABS_ERROR": float(metrics["sum_abs_error"]),
        "SUM_SQ_ERROR": float(metrics["sum_sq_error"]),
        "SUM_Y": float(metrics["sum_y"]),
        "SUM_Y2": float(metrics["sum_y2"]),

        "PEAK_PROCESS_MEMORY_MB": float(metrics["peak_process_memory_mb"]),
        "PEAK_GPU_MEMORY_MB": float(metrics["peak_gpu_memory_mb"]),

        "ARTIFACT_LOCATION": model_dir,
        "NOTES": notes,
    }

    save_pytorch_result(result)

    print("\nSaved PyTorch result:")
    print(result)

    return result


In [ ]:
# GPU smoke test.
pt_smoke_gpu = run_pytorch_ddp(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    use_gpu=True,
    num_nodes=1,
    workers_per_node=1,
    dataset=None,
    train_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=1.0,
    scaling_label="pytorch_gpu_baseline_1_node_smoke_1pct_train_1pct_test",
    epochs=1,
    batch_size=4096,
    hidden_size=64,
    notes="Smoke test for Snowflake PyTorch DDP on GPU. 1% train and 1% test."
)


In [ ]:
pt_gpu_base_1node_full = run_pytorch_ddp(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    use_gpu=True,
    num_nodes=1,
    workers_per_node=1,
    dataset=None,
    train_sample_pct=100.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="pytorch_gpu_baseline_1_node_full_train_full_test",
    epochs=1,
    batch_size=8192,
    hidden_size=64,
    notes="Snowflake PyTorch DDP GPU baseline. 1 node. Full train and full test."
)

In [ ]:
pt_gpu_base_1node_full_e3 = run_pytorch_ddp(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    use_gpu=True,
    num_nodes=1,
    workers_per_node=1,
    dataset=None,
    train_sample_pct=100.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="pytorch_gpu_baseline_1_node_full_train_full_test_e10",
    epochs=10,
    batch_size=8192,
    hidden_size=64,
    notes="Snowflake PyTorch DDP GPU baseline. 1 node. Full train/test. 10 epochs."
)

In [ ]:
import ray
from snowflake.ml.runtime_cluster import scale_cluster, get_nodes, get_ray_dashboard_url

ray.init(address="auto", ignore_reinit_error=True)

def print_ray_status():
    print("Current Ray nodes:", len(get_nodes()))
    print("Ray cluster resources:", ray.cluster_resources())
    print("Ray available resources:", ray.available_resources())

    try:
        print("Ray dashboard:", get_ray_dashboard_url())
    except Exception as e:
        print("Dashboard unavailable:", e)


def scale_ray_cluster_to(n_nodes):
    print(f"Scaling Ray cluster to {n_nodes} node(s)...")

    scale_cluster(
        expected_cluster_size=n_nodes,
        options={
            "rollback_after_seconds": 1800,
            "block_until_min_cluster_size": n_nodes,
        }
    )

    print_ray_status()


print_ray_status()

In [ ]:
scale_ray_cluster_to(2)

In [ ]:
import ray
import time

try:
    ray.shutdown()
    print("Ray shutdown called.")
except Exception as e:
    print("ray.shutdown error:", repr(e))

time.sleep(10)

ray.init(address="auto", ignore_reinit_error=True)

print("Cluster resources:")
print(ray.cluster_resources())

print("Available resources:")
print(ray.available_resources())

In [ ]:
pt_gpu_base_2nodes_full_e10 = run_pytorch_ddp(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    use_gpu=True,
    num_nodes=2,
    workers_per_node=1,
    dataset=None,
    train_sample_pct=100.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="pytorch_gpu_baseline_2_nodes_full_train_full_test_e10",
    epochs=10,
    batch_size=8192,
    hidden_size=64,
    notes="Snowflake PyTorch DDP GPU baseline. 2 nodes. Full train/test. 10 epochs."
)

In [ ]:
scale_ray_cluster_to(4)

print(ray.cluster_resources())
print(ray.available_resources())

In [ ]:
pt_gpu_base_4nodes_full_e10 = run_pytorch_ddp(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    use_gpu=True,
    num_nodes=4,
    workers_per_node=1,
    dataset=None,
    train_sample_pct=100.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="pytorch_gpu_baseline_4_nodes_full_train_full_test_e10",
    epochs=10,
    batch_size=8192,
    hidden_size=64,
    notes="Snowflake PyTorch DDP GPU baseline. 4 nodes. Full train/test. 10 epochs."
)

In [ ]:
pt_gpu_aug_4nodes_full_e10 = run_pytorch_ddp(
    features=AUGMENTED_FEATURES,
    feature_set_name="augmented_with_t5",
    use_gpu=True,
    num_nodes=4,
    workers_per_node=1,
    dataset=None,
    train_sample_pct=100.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="pytorch_gpu_augmented_4_nodes_full_train_full_test_e10",
    epochs=10,
    batch_size=8192,
    hidden_size=64,
    notes="Snowflake PyTorch DDP GPU augmented. 4 nodes. Full train/test. 10 epochs."
)

In [ ]:
scale_ray_cluster_to(1)

In [ ]:
scale_ray_cluster_to(2)

In [ ]:
pt_gpu_sep_yellow_2nodes_e10 = run_pytorch_ddp(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    use_gpu=True,
    num_nodes=2,
    workers_per_node=1,
    dataset="yellow",
    train_sample_pct=100.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="pytorch_gpu_separate_yellow_2_nodes_full_train_full_test_e10",
    epochs=10,
    batch_size=8192,
    hidden_size=64,
    notes="Snowflake PyTorch DDP separate yellow model. 2 nodes. Full train/test. 10 epochs."
)

In [ ]:
pt_gpu_sep_green_2nodes_e10 = run_pytorch_ddp(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    use_gpu=True,
    num_nodes=2,
    workers_per_node=1,
    dataset="green",
    train_sample_pct=100.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="pytorch_gpu_separate_green_2_nodes_full_train_full_test_e10",
    epochs=10,
    batch_size=8192,
    hidden_size=64,
    notes="Snowflake PyTorch DDP separate green model. 2 nodes. Full train/test. 10 epochs."
)

In [ ]:
pt_gpu_sep_fhv_2nodes_e10 = run_pytorch_ddp(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    use_gpu=True,
    num_nodes=2,
    workers_per_node=1,
    dataset="fhv",
    train_sample_pct=100.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="pytorch_gpu_separate_fhv_2_nodes_full_train_full_test_e10",
    epochs=10,
    batch_size=8192,
    hidden_size=64,
    notes="Snowflake PyTorch DDP separate fhv model. 2 nodes. Full train/test. 10 epochs."
)

In [ ]:
pt_gpu_sep_fhvhv_2nodes_e10 = run_pytorch_ddp(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    use_gpu=True,
    num_nodes=2,
    workers_per_node=1,
    dataset="fhvhv",
    train_sample_pct=100.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="pytorch_gpu_separate_fhvhv_2_nodes_full_train_full_test_e10",
    epochs=10,
    batch_size=8192,
    hidden_size=64,
    notes="Snowflake PyTorch DDP separate fhvhv model. 2 nodes. Full train/test. 10 epochs."
)

In [ ]:
%%sql -r dataframe_7
SELECT
    SUM(train_rows) AS train_rows,
    SUM(metric_rows) AS metric_rows,
    SUM(train_seconds) AS train_seconds,
    SUM(predict_seconds) AS predict_seconds,
    SUM(sum_abs_error) / SUM(metric_rows) AS mae,
    SQRT(SUM(sum_sq_error) / SUM(metric_rows)) AS rmse,
    1 - (
        SUM(sum_sq_error)
        /
        (
            SUM(sum_y2) - SUM(metric_rows) * POWER(SUM(sum_y) / SUM(metric_rows), 2)
        )
    ) AS r2,
    MAX(peak_process_memory_mb) AS peak_process_memory_mb,
    MAX(peak_gpu_memory_mb) AS peak_gpu_memory_mb
FROM GOLD.T7_T10_PYTORCH_RESULTS
WHERE scaling_label IN (
    'pytorch_gpu_separate_yellow_2_nodes_full_train_full_test_e10',
    'pytorch_gpu_separate_green_2_nodes_full_train_full_test_e10',
    'pytorch_gpu_separate_fhv_2_nodes_full_train_full_test_e10',
    'pytorch_gpu_separate_fhvhv_2_nodes_full_train_full_test_e10'
);

In [ ]:
SGD_RESULT_COLUMNS_NO_CREATED = [
    "RUN_ID",
    "TASK",
    "MODEL_NAME",
    "MODEL_SCOPE",
    "DATASET",
    "FEATURE_SET",
    "SCALING_LABEL",

    "COMPUTE_TYPE",
    "TRAIN_SPLIT",
    "METRIC_SPLIT",
    "TRAIN_SAMPLE_PCT",
    "METRIC_SAMPLE_PCT",

    "TRAIN_ROWS",
    "METRIC_ROWS",

    "TRAIN_SECONDS",
    "PREDICT_SECONDS",

    "TRAIN_LOSS",
    "MAE",
    "RMSE",
    "R2",
    "LOG_RMSE",

    "SUM_ABS_ERROR",
    "SUM_SQ_ERROR",
    "SUM_Y",
    "SUM_Y2",

    "PEAK_PROCESS_MEMORY_MB",
    "PEAK_GPU_MEMORY_MB",

    "WAREHOUSE_NAME",
    "WAREHOUSE_SIZE",

    "NOTES",
]


def save_sgd_result(result):
    cols = ", ".join(SGD_RESULT_COLUMNS_NO_CREATED)
    vals = ", ".join(sql_value(result.get(c)) for c in SGD_RESULT_COLUMNS_NO_CREATED)

    session.sql(f"""
        INSERT INTO GOLD.T7_SGD_RESULTS ({cols})
        VALUES ({vals})
    """).collect()


In [ ]:
def make_sgd_sdf(features, split, dataset=None, sample_pct=100.0):
    select_exprs = []

    for f in features:
        select_exprs.append(f"COALESCE({f}, 0)::FLOAT AS {f}")

    select_exprs.append(f"COALESCE({TARGET_LOG}, 0)::FLOAT AS {TARGET_LOG}")
    select_exprs.append(f"COALESCE({TARGET_RAW}, 0)::FLOAT AS {TARGET_RAW}")

    sql = f"""
        SELECT
            {", ".join(select_exprs)}
        FROM {TABLE}
        WHERE SPLIT = '{split}'
        {dataset_filter_sql(dataset)}
        {sample_condition_sql(sample_pct)}
    """

    return session.sql(sql)


def run_sgd_partial_fit(
    features,
    feature_set_name,
    dataset=None,
    train_sample_pct=100.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="sgd_partial_fit",
    epochs=1,
    notes="",
):
    run_id = str(uuid.uuid4())

    model_scope = "global" if dataset is None else "separate"
    dataset_label = "all" if dataset is None else dataset

    warehouse_name, warehouse_size = get_warehouse_info()

    print("=" * 80)
    print(f"Run: {scaling_label}")
    print(f"Feature set: {feature_set_name}")
    print(f"Dataset: {dataset_label}")
    print(f"Train sample: {train_sample_pct}%")
    print(f"Metric sample: {metric_sample_pct}%")
    print(f"Epochs: {epochs}")
    print("=" * 80)

    print("Computing scaler statistics in Snowflake...")
    mean_values, std_values = get_scaler_stats_sql(
        features=features,
        dataset=dataset,
        train_sample_pct=train_sample_pct,
    )

    model = SGDRegressor(
        loss="huber",
        epsilon=0.1,
        penalty="l2",
        alpha=0.01,
        learning_rate="constant",
        eta0=0.00001,
        average=True,
        random_state=42
    )

    train_sdf = make_sgd_sdf(
        features=features,
        split="train",
        dataset=dataset,
        sample_pct=train_sample_pct,
    )

    expected_train_rows = train_sdf.count()
    print(f"Expected train rows: {expected_train_rows:,}")

    train_rows = 0
    train_loss_sum = 0.0
    train_batches = 0

    with ResourceMonitor(interval=1.0) as monitor:
        start_train = time.time()

        for epoch in range(epochs):
            print(f"Training epoch {epoch + 1}/{epochs}...")

            # Re-create the Snowpark dataframe for each epoch to get a fresh iterator.
            train_sdf_epoch = make_sgd_sdf(
                features=features,
                split="train",
                dataset=dataset,
                sample_pct=train_sample_pct,
            )

            for i, pdf in enumerate(train_sdf_epoch.to_pandas_batches(), start=1):
                pdf = pdf.fillna(0)

                X = pdf[features].astype("float32").values
                y = pdf[TARGET_LOG].astype("float32").values

                X_scaled = (X - mean_values) / std_values

                model.partial_fit(X_scaled, y)

                pred_train = model.predict(X_scaled)
                train_errors = y - pred_train
                train_loss_sum += float((train_errors ** 2).sum())

                train_rows += len(pdf)
                train_batches += 1

                if train_batches % 25 == 0:
                    print(f"Training batches processed: {train_batches}, rows={train_rows:,}")

        train_seconds = time.time() - start_train

    train_loss = train_loss_sum / train_rows if train_rows > 0 else None

    print(f"Training done. Rows processed: {train_rows:,}, seconds: {train_seconds:.2f}")

    metric_sdf = make_sgd_sdf(
        features=features,
        split=metric_split,
        dataset=dataset,
        sample_pct=metric_sample_pct,
    )

    expected_metric_rows = metric_sdf.count()
    print(f"Expected metric rows: {expected_metric_rows:,}")

    n = 0
    sum_abs_error = 0.0
    sum_sq_error = 0.0
    sum_y = 0.0
    sum_y2 = 0.0
    sum_log_sq_error = 0.0

    start_pred = time.time()

    for i, pdf in enumerate(metric_sdf.to_pandas_batches(), start=1):
        pdf = pdf.fillna(0)

        X = pdf[features].astype("float32").values
        y_true = pdf[TARGET_RAW].astype("float64").values
        y_true_log = pdf[TARGET_LOG].astype("float64").values

        X_scaled = (X - mean_values) / std_values

        pred_log = model.predict(X_scaled)
        pred_log = np.clip(pred_log, 0, np.log1p(5000))

        pred_raw = np.expm1(pred_log)
        pred_raw = np.clip(pred_raw, 0, None)

        errors = y_true - pred_raw
        log_errors = y_true_log - pred_log

        n_batch = len(y_true)
        n += n_batch

        sum_abs_error += float(np.abs(errors).sum())
        sum_sq_error += float((errors ** 2).sum())
        sum_y += float(y_true.sum())
        sum_y2 += float((y_true ** 2).sum())
        sum_log_sq_error += float((log_errors ** 2).sum())

        if i % 25 == 0:
            print(f"Evaluation batches processed: {i}, rows={n:,}")

    predict_seconds = time.time() - start_pred

    mae, rmse, r2 = compute_metrics_from_sums(n, sum_abs_error, sum_sq_error, sum_y, sum_y2)
    log_rmse = (sum_log_sq_error / n) ** 0.5

    result = {
        "RUN_ID": run_id,
        "TASK": "T7",
        "MODEL_NAME": "SGDRegressor_partial_fit",
        "MODEL_SCOPE": model_scope,
        "DATASET": dataset_label,
        "FEATURE_SET": feature_set_name,
        "SCALING_LABEL": scaling_label,

        "COMPUTE_TYPE": "Snowflake notebook CPU batched partial_fit",
        "TRAIN_SPLIT": "train",
        "METRIC_SPLIT": metric_split,
        "TRAIN_SAMPLE_PCT": float(train_sample_pct),
        "METRIC_SAMPLE_PCT": float(metric_sample_pct),

        "TRAIN_ROWS": int(train_rows),
        "METRIC_ROWS": int(n),

        "TRAIN_SECONDS": float(train_seconds),
        "PREDICT_SECONDS": float(predict_seconds),

        "TRAIN_LOSS": None if train_loss is None else float(train_loss),
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": None if r2 is None else float(r2),
        "LOG_RMSE": float(log_rmse),

        "SUM_ABS_ERROR": float(sum_abs_error),
        "SUM_SQ_ERROR": float(sum_sq_error),
        "SUM_Y": float(sum_y),
        "SUM_Y2": float(sum_y2),

        "PEAK_PROCESS_MEMORY_MB": float(monitor.peak_process_memory_mb),
        "PEAK_GPU_MEMORY_MB": float(monitor.peak_gpu_memory_mb),

        "WAREHOUSE_NAME": warehouse_name,
        "WAREHOUSE_SIZE": warehouse_size,

        "NOTES": notes,
    }

    save_sgd_result(result)

    print("\nSaved SGD result:")
    print(result)

    return model, result


In [ ]:
# SGD smoke test: 1% train and 1% test.

sgd_smoke_model, sgd_smoke_result = run_sgd_partial_fit(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    dataset=None,
    train_sample_pct=1.0,
    metric_split="test",
    metric_sample_pct=1.0,
    scaling_label="sgd_baseline_smoke_1pct_train_1pct_test",
    epochs=1,
    notes="Smoke test for SGDRegressor partial_fit. 1% train and 1% test."
)


In [ ]:
sgd_base_full_e1_model, sgd_base_full_e1_result = run_sgd_partial_fit(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    dataset=None,
    train_sample_pct=100.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="sgd_baseline_full_train_full_test_e1",
    epochs=1,
    notes="SGDRegressor partial_fit baseline. Full train and full test. 1 epoch."
)

In [ ]:
sgd_base_full_stable_e1_model, sgd_base_full_stable_e1_result = run_sgd_partial_fit(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    dataset=None,
    train_sample_pct=100.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="sgd_baseline_full_train_full_test_stable_e1",
    epochs=1,
    notes="Stable SGDRegressor partial_fit baseline. Huber loss, small learning rate. Full train/test. 1 epoch."
)

In [ ]:
sgd_base_full_stable_e10_model, sgd_base_full_stable_e10_result = run_sgd_partial_fit(
    features=BASELINE_FEATURES,
    feature_set_name="baseline_without_augmentation",
    dataset=None,
    train_sample_pct=100.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="sgd_baseline_full_train_full_test_stable_e10",
    epochs=10,
    notes="Stable SGDRegressor partial_fit baseline. Huber loss, small learning rate. Full train/test. 10 epochs."
)

In [ ]:
sgd_aug_full_stable_e1_model, sgd_aug_full_stable_e1_result = run_sgd_partial_fit(
    features=AUGMENTED_FEATURES,
    feature_set_name="augmented_with_t5",
    dataset=None,
    train_sample_pct=100.0,
    metric_split="test",
    metric_sample_pct=100.0,
    scaling_label="sgd_augmented_full_train_full_test_stable_e1",
    epochs=1,
    notes="Stable SGDRegressor partial_fit with T5 augmentation. Huber loss, small learning rate. Full train/test. 1 epoch."
)

In [ ]:
%%sql -r dataframe_8
INSERT INTO GOLD.T7_T10_PYTORCH_RESULTS (
    RUN_ID,
    TASK,
    MODEL_NAME,
    MODEL_SCOPE,
    DATASET,
    FEATURE_SET,
    SCALING_LABEL,
    COMPUTE_TYPE,
    USE_GPU,
    NUM_NODES,
    WORKERS_PER_NODE,
    TOTAL_WORKERS,
    EPOCHS,
    BATCH_SIZE,
    TRAIN_SPLIT,
    METRIC_SPLIT,
    TRAIN_SAMPLE_PCT,
    METRIC_SAMPLE_PCT,
    TRAIN_ROWS,
    METRIC_ROWS,
    TRAIN_SECONDS,
    PREDICT_SECONDS,
    TRAIN_LOSS,
    MAE,
    RMSE,
    R2,
    LOG_RMSE,
    SUM_ABS_ERROR,
    SUM_SQ_ERROR,
    SUM_Y,
    SUM_Y2,
    PEAK_PROCESS_MEMORY_MB,
    PEAK_GPU_MEMORY_MB,
    ARTIFACT_LOCATION,
    NOTES
)
SELECT
    UUID_STRING() AS RUN_ID,
    'T10' AS TASK,
    'Combined_Separate_PyTorch_DDP_Models' AS MODEL_NAME,
    'separate_combined' AS MODEL_SCOPE,
    'yellow+green+fhv+fhvhv' AS DATASET,
    'baseline_without_augmentation' AS FEATURE_SET,
    'pytorch_gpu_combined_separate_2_nodes_full_train_full_test_e10' AS SCALING_LABEL,
    'Snowflake PyTorch DDP GPU' AS COMPUTE_TYPE,
    TRUE AS USE_GPU,
    2 AS NUM_NODES,
    1 AS WORKERS_PER_NODE,
    2 AS TOTAL_WORKERS,
    10 AS EPOCHS,
    8192 AS BATCH_SIZE,
    'train' AS TRAIN_SPLIT,
    'test' AS METRIC_SPLIT,
    100.0 AS TRAIN_SAMPLE_PCT,
    100.0 AS METRIC_SAMPLE_PCT,

    SUM(TRAIN_ROWS) AS TRAIN_ROWS,
    SUM(METRIC_ROWS) AS METRIC_ROWS,
    SUM(TRAIN_SECONDS) AS TRAIN_SECONDS,
    SUM(PREDICT_SECONDS) AS PREDICT_SECONDS,

    SUM(TRAIN_LOSS * TRAIN_ROWS) / SUM(TRAIN_ROWS) AS TRAIN_LOSS,

    SUM(SUM_ABS_ERROR) / SUM(METRIC_ROWS) AS MAE,
    SQRT(SUM(SUM_SQ_ERROR) / SUM(METRIC_ROWS)) AS RMSE,
    1 - (
        SUM(SUM_SQ_ERROR)
        /
        (
            SUM(SUM_Y2) - SUM(METRIC_ROWS) * POWER(SUM(SUM_Y) / SUM(METRIC_ROWS), 2)
        )
    ) AS R2,
    NULL AS LOG_RMSE,

    SUM(SUM_ABS_ERROR) AS SUM_ABS_ERROR,
    SUM(SUM_SQ_ERROR) AS SUM_SQ_ERROR,
    SUM(SUM_Y) AS SUM_Y,
    SUM(SUM_Y2) AS SUM_Y2,

    MAX(PEAK_PROCESS_MEMORY_MB) AS PEAK_PROCESS_MEMORY_MB,
    MAX(PEAK_GPU_MEMORY_MB) AS PEAK_GPU_MEMORY_MB,

    NULL AS ARTIFACT_LOCATION,
    'Aggregated result from four separate PyTorch DDP service-specific models: yellow, green, fhv, fhvhv.' AS NOTES
FROM GOLD.T7_T10_PYTORCH_RESULTS
WHERE SCALING_LABEL IN (
    'pytorch_gpu_separate_yellow_2_nodes_full_train_full_test_e10',
    'pytorch_gpu_separate_green_2_nodes_full_train_full_test_e10',
    'pytorch_gpu_separate_fhv_2_nodes_full_train_full_test_e10',
    'pytorch_gpu_separate_fhvhv_2_nodes_full_train_full_test_e10'
);

In [ ]:
import os
from snowflake.snowpark.context import get_active_session

session = get_active_session()

OUT_DIR = "/tmp/t7_results"
os.makedirs(OUT_DIR, exist_ok=True)


def export_query_to_csv(query, filename):
    path = os.path.join(OUT_DIR, filename)
    pdf = session.sql(query).to_pandas()
    pdf.to_csv(path, index=False)
    print(f"Saved {len(pdf):,} rows to {path}")
    return path

In [ ]:
xgb_csv = export_query_to_csv("""
SELECT
    TASK,
    MODEL_NAME,
    MODEL_SCOPE,
    DATASET,
    FEATURE_SET,
    SCALING_LABEL,
    TRAIN_ROWS,
    METRIC_ROWS,
    RAY_NODES,
    N_ESTIMATORS,
    TRAIN_SECONDS,
    PREDICT_SECONDS,
    MAE,
    RMSE,
    R2,
    PEAK_PROCESS_MEMORY_MB,
    PEAK_GPU_MEMORY_MB,
    CREATED_AT
FROM GOLD.T7_T10_XGB_RESULTS
ORDER BY CREATED_AT
""", "t7_xgboost_results.csv")

In [ ]:
pytorch_csv = export_query_to_csv("""
SELECT
    TASK,
    MODEL_NAME,
    MODEL_SCOPE,
    DATASET,
    FEATURE_SET,
    SCALING_LABEL,
    COMPUTE_TYPE,
    USE_GPU,
    NUM_NODES,
    WORKERS_PER_NODE,
    TOTAL_WORKERS,
    EPOCHS,
    BATCH_SIZE,
    TRAIN_ROWS,
    METRIC_ROWS,
    TRAIN_SECONDS,
    PREDICT_SECONDS,
    TRAIN_LOSS,
    MAE,
    RMSE,
    R2,
    LOG_RMSE,
    PEAK_PROCESS_MEMORY_MB,
    PEAK_GPU_MEMORY_MB,
    ARTIFACT_LOCATION,
    CREATED_AT
FROM GOLD.T7_T10_PYTORCH_RESULTS
ORDER BY CREATED_AT
""", "t7_pytorch_results.csv")

In [ ]:
sgd_csv = export_query_to_csv("""
SELECT
    TASK,
    MODEL_NAME,
    MODEL_SCOPE,
    DATASET,
    FEATURE_SET,
    SCALING_LABEL,
    COMPUTE_TYPE,
    TRAIN_ROWS,
    METRIC_ROWS,
    TRAIN_SECONDS,
    PREDICT_SECONDS,
    TRAIN_LOSS,
    MAE,
    RMSE,
    R2,
    LOG_RMSE,
    PEAK_PROCESS_MEMORY_MB,
    PEAK_GPU_MEMORY_MB,
    WAREHOUSE_NAME,
    WAREHOUSE_SIZE,
    CREATED_AT
FROM GOLD.T7_SGD_RESULTS
ORDER BY CREATED_AT
""", "t7_sgd_results.csv")

In [ ]:
combined_csv = export_query_to_csv("""
SELECT
    'XGBoost' AS ALGORITHM,
    TASK,
    MODEL_NAME,
    MODEL_SCOPE,
    DATASET,
    FEATURE_SET,
    SCALING_LABEL,
    'distributed_xgboost' AS ALGORITHM_TYPE,
    RAY_NODES AS WORKERS,
    N_ESTIMATORS AS ITERATIONS,
    TRAIN_ROWS,
    METRIC_ROWS,
    TRAIN_SECONDS,
    PREDICT_SECONDS,
    NULL AS TRAIN_LOSS,
    MAE,
    RMSE,
    R2,
    NULL AS LOG_RMSE,
    PEAK_PROCESS_MEMORY_MB,
    PEAK_GPU_MEMORY_MB,
    CREATED_AT
FROM GOLD.T7_T10_XGB_RESULTS

UNION ALL

SELECT
    'PyTorch DDP' AS ALGORITHM,
    TASK,
    MODEL_NAME,
    MODEL_SCOPE,
    DATASET,
    FEATURE_SET,
    SCALING_LABEL,
    'distributed_neural_network' AS ALGORITHM_TYPE,
    TOTAL_WORKERS AS WORKERS,
    EPOCHS AS ITERATIONS,
    TRAIN_ROWS,
    METRIC_ROWS,
    TRAIN_SECONDS,
    PREDICT_SECONDS,
    TRAIN_LOSS,
    MAE,
    RMSE,
    R2,
    LOG_RMSE,
    PEAK_PROCESS_MEMORY_MB,
    PEAK_GPU_MEMORY_MB,
    CREATED_AT
FROM GOLD.T7_T10_PYTORCH_RESULTS

UNION ALL

SELECT
    'SGD partial_fit' AS ALGORITHM,
    TASK,
    MODEL_NAME,
    MODEL_SCOPE,
    DATASET,
    FEATURE_SET,
    SCALING_LABEL,
    'sklearn_partial_fit' AS ALGORITHM_TYPE,
    1 AS WORKERS,
    NULL AS ITERATIONS,
    TRAIN_ROWS,
    METRIC_ROWS,
    TRAIN_SECONDS,
    PREDICT_SECONDS,
    TRAIN_LOSS,
    MAE,
    RMSE,
    R2,
    LOG_RMSE,
    PEAK_PROCESS_MEMORY_MB,
    PEAK_GPU_MEMORY_MB,
    CREATED_AT
FROM GOLD.T7_SGD_RESULTS

ORDER BY CREATED_AT
""", "t7_all_model_results_combined.csv")

In [ ]:
summary_csv = export_query_to_csv("""
SELECT *
FROM (
    SELECT
        'XGBoost' AS ALGORITHM,
        MODEL_NAME,
        MODEL_SCOPE,
        DATASET,
        FEATURE_SET,
        SCALING_LABEL,
        RAY_NODES AS WORKERS,
        N_ESTIMATORS AS ITERATIONS,
        TRAIN_ROWS,
        METRIC_ROWS,
        TRAIN_SECONDS,
        PREDICT_SECONDS,
        MAE,
        RMSE,
        R2,
        PEAK_PROCESS_MEMORY_MB,
        PEAK_GPU_MEMORY_MB,
        CREATED_AT
    FROM GOLD.T7_T10_XGB_RESULTS
    WHERE SCALING_LABEL IN (
        'xgb_gpu_baseline_2_nodes_full_train_full_test',
        'xgb_gpu_baseline_4_nodes_full_train_full_test',
        'xgb_gpu_augmented_4_nodes_full_train_full_test',
        'xgb_gpu_combined_separate_4_nodes_full_train_full_test',
        'xgb_cpu_augmented_4_nodes_full_train_full_test'
    )

    UNION ALL

    SELECT
        'PyTorch DDP' AS ALGORITHM,
        MODEL_NAME,
        MODEL_SCOPE,
        DATASET,
        FEATURE_SET,
        SCALING_LABEL,
        TOTAL_WORKERS AS WORKERS,
        EPOCHS AS ITERATIONS,
        TRAIN_ROWS,
        METRIC_ROWS,
        TRAIN_SECONDS,
        PREDICT_SECONDS,
        MAE,
        RMSE,
        R2,
        PEAK_PROCESS_MEMORY_MB,
        PEAK_GPU_MEMORY_MB,
        CREATED_AT
    FROM GOLD.T7_T10_PYTORCH_RESULTS
    WHERE SCALING_LABEL IN (
        'pytorch_gpu_baseline_1_node_full_train_full_test_e10',
        'pytorch_gpu_baseline_2_nodes_full_train_full_test_e10',
        'pytorch_gpu_baseline_4_nodes_full_train_full_test_e10',
        'pytorch_gpu_augmented_4_nodes_full_train_full_test_e10',
        'pytorch_gpu_combined_separate_2_nodes_full_train_full_test_e10'
    )

    UNION ALL

    SELECT
        'SGD partial_fit' AS ALGORITHM,
        MODEL_NAME,
        MODEL_SCOPE,
        DATASET,
        FEATURE_SET,
        SCALING_LABEL,
        1 AS WORKERS,
        NULL AS ITERATIONS,
        TRAIN_ROWS,
        METRIC_ROWS,
        TRAIN_SECONDS,
        PREDICT_SECONDS,
        MAE,
        RMSE,
        R2,
        PEAK_PROCESS_MEMORY_MB,
        PEAK_GPU_MEMORY_MB,
        CREATED_AT
    FROM GOLD.T7_SGD_RESULTS
    WHERE SCALING_LABEL IN (
        'sgd_baseline_full_train_full_test_stable_e1',
        'sgd_augmented_full_train_full_test_stable_e1',
        'sgd_baseline_full_train_full_test_stable_e10'
    )
)
ORDER BY ALGORITHM, CREATED_AT
""", "t7_report_summary.csv")

In [ ]:
print("CSV files created:")
print(xgb_csv)
print(pytorch_csv)
print(sgd_csv)
print(combined_csv)
print(summary_csv)

In [ ]:
session.sql("CREATE STAGE IF NOT EXISTS GOLD.T7_RESULTS_STAGE").collect()

for filename in os.listdir(OUT_DIR):
    if filename.endswith(".csv"):
        local_path = os.path.join(OUT_DIR, filename)
        put_result = session.file.put(
            local_path,
            "@GOLD.T7_RESULTS_STAGE",
            overwrite=True,
            auto_compress=False
        )
        print(filename, put_result)

In [ ]:
%%sql -r dataframe_9
LIST @GOLD.T7_RESULTS_STAGE;

In [ ]:
%%sql -r dataframe_4
SELECT
    task,
    model_name,
    model_scope,
    dataset,
    feature_set,
    scaling_label,
    train_rows,
    metric_rows,
    ray_nodes,
    n_estimators,
    train_seconds,
    predict_seconds,
    mae,
    rmse,
    r2,
    peak_process_memory_mb,
    peak_gpu_memory_mb,
    created_at
FROM GOLD.T7_T10_XGB_RESULTS
ORDER BY created_at;

In [ ]:
SELECT
    TASK,
    MODEL_NAME,
    MODEL_SCOPE,
    DATASET,
    FEATURE_SET,
    SCALING_LABEL,
    COMPUTE_TYPE,
    USE_GPU,
    NUM_NODES,
    WORKERS_PER_NODE,
    TOTAL_WORKERS,
    EPOCHS,
    BATCH_SIZE,
    TRAIN_ROWS,
    METRIC_ROWS,
    TRAIN_SECONDS,
    PREDICT_SECONDS,
    TRAIN_LOSS,
    MAE,
    RMSE,
    R2,
    LOG_RMSE,
    PEAK_PROCESS_MEMORY_MB,
    PEAK_GPU_MEMORY_MB,
    ARTIFACT_LOCATION,
    CREATED_AT
FROM GOLD.T7_T10_PYTORCH_RESULTS
ORDER BY CREATED_AT;

In [ ]:
SELECT
    TASK,
    MODEL_NAME,
    MODEL_SCOPE,
    DATASET,
    FEATURE_SET,
    SCALING_LABEL,
    COMPUTE_TYPE,
    TRAIN_ROWS,
    METRIC_ROWS,
    TRAIN_SECONDS,
    PREDICT_SECONDS,
    TRAIN_LOSS,
    MAE,
    RMSE,
    R2,
    LOG_RMSE,
    PEAK_PROCESS_MEMORY_MB,
    PEAK_GPU_MEMORY_MB,
    WAREHOUSE_NAME,
    WAREHOUSE_SIZE,
    CREATED_AT
FROM GOLD.T7_SGD_RESULTS
ORDER BY CREATED_AT;

In [ ]:
%%sql -r dataframe_6
SELECT *
FROM (
    SELECT
        'XGBoost' AS ALGORITHM,
        MODEL_NAME,
        MODEL_SCOPE,
        DATASET,
        FEATURE_SET,
        SCALING_LABEL,
        RAY_NODES AS WORKERS,
        N_ESTIMATORS AS ITERATIONS,
        TRAIN_ROWS,
        METRIC_ROWS,
        TRAIN_SECONDS,
        PREDICT_SECONDS,
        MAE,
        RMSE,
        R2,
        PEAK_PROCESS_MEMORY_MB,
        PEAK_GPU_MEMORY_MB,
        CREATED_AT
    FROM GOLD.T7_T10_XGB_RESULTS
    WHERE SCALING_LABEL IN (
        'xgb_gpu_baseline_2_nodes_full_train_full_test',
        'xgb_gpu_baseline_4_nodes_full_train_full_test',
        'xgb_gpu_augmented_4_nodes_full_train_full_test',
        'xgb_gpu_combined_separate_4_nodes_full_train_full_test',
        'xgb_cpu_augmented_4_nodes_full_train_full_test'
    )

    UNION ALL

    SELECT
        'PyTorch DDP' AS ALGORITHM,
        MODEL_NAME,
        MODEL_SCOPE,
        DATASET,
        FEATURE_SET,
        SCALING_LABEL,
        TOTAL_WORKERS AS WORKERS,
        EPOCHS AS ITERATIONS,
        TRAIN_ROWS,
        METRIC_ROWS,
        TRAIN_SECONDS,
        PREDICT_SECONDS,
        MAE,
        RMSE,
        R2,
        PEAK_PROCESS_MEMORY_MB,
        PEAK_GPU_MEMORY_MB,
        CREATED_AT
    FROM GOLD.T7_T10_PYTORCH_RESULTS
    WHERE SCALING_LABEL IN (
        'pytorch_gpu_baseline_1_node_full_train_full_test_e10',
        'pytorch_gpu_baseline_2_nodes_full_train_full_test_e10',
        'pytorch_gpu_baseline_4_nodes_full_train_full_test_e10',
        'pytorch_gpu_augmented_4_nodes_full_train_full_test_e10',
        'pytorch_gpu_combined_separate_2_nodes_full_train_full_test_e10'
    )

    UNION ALL

    SELECT
        'SGD partial_fit' AS ALGORITHM,
        MODEL_NAME,
        MODEL_SCOPE,
        DATASET,
        FEATURE_SET,
        SCALING_LABEL,
        1 AS WORKERS,
        NULL AS ITERATIONS,
        TRAIN_ROWS,
        METRIC_ROWS,
        TRAIN_SECONDS,
        PREDICT_SECONDS,
        MAE,
        RMSE,
        R2,
        PEAK_PROCESS_MEMORY_MB,
        PEAK_GPU_MEMORY_MB,
        CREATED_AT
    FROM GOLD.T7_SGD_RESULTS
    WHERE SCALING_LABEL IN (
        'sgd_baseline_full_train_full_test_stable_e1',
        'sgd_augmented_full_train_full_test_stable_e1',
        'sgd_baseline_full_train_full_test_stable_e10'
    )
)
ORDER BY ALGORITHM, CREATED_AT;